# Stage 11: independent multi-cut delta-U surface baseline

This standalone Colab notebook starts the independent 5-point track. It creates four pseudo-test cuts per training well and cross-fits a low-frequency `U = TVT + Z` surface model without public predictions. It evaluates ordinary well folds, geographic blocks, and typewell-signature holdouts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Runtime

A GPU is not used in Stage 11. A standard or high-RAM CPU session is preferable. The full run reads 773 wells, creates the fixed multi-cut cache, and trains small HGB models for three holdout families.


In [ ]:
import os, platform
print('Python runtime:', platform.python_version())
print('CPU count:', os.cpu_count())
try:
    import psutil
    print('RAM GiB:', round(psutil.virtual_memory().total / 2**30, 1))
except ImportError:
    pass


## Run Stage 11

Keep `LIMIT_WELLS = None` for the decision run. Use 24 only for a wiring smoke test. A smoke test must not be used for model promotion.


In [ ]:
RUN_ID = 'stage11_multicut_delta_u_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-delta-u',
        '--config', 'configs/experiment/stage11_multicut_delta_u.yaml',
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
        print('\n'.join(tail))
        raise RuntimeError(f'Stage 11 failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Validation summary

This experiment does not create a Kaggle submission. `promoted_to_alignment_benchmark` authorizes the raw-NCC and learned-emission Stage 12 work; it does not claim that the surface model is already a 5-point solution.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
{
    'promoted_to_alignment_benchmark': summary['promoted_to_alignment_benchmark'],
    'n_wells': summary['n_wells'],
    'n_cuts': summary['n_cuts'],
    'feature_count': summary['feature_count'],
    'hidden_target_invariance': summary['hidden_target_invariance']['passed'],
    'base_rmse': summary['standard']['base_metrics']['pooled_rmse'],
    'candidate_rmse': summary['standard']['candidate_metrics']['pooled_rmse'],
    'rmse_delta': summary['standard']['pooled_rmse_delta'],
    'spatial_delta': summary['spatial']['pooled_rmse_delta'],
    'typewell_delta': summary['typewell']['pooled_rmse_delta'],
    'improved_folds': f"{summary['improved_folds']}/{len(summary['standard']['fold_deltas'])}",
    'bootstrap_95pct': [summary['bootstrap']['ci_2_5'], summary['bootstrap']['ci_97_5']],
    'fixed_correction_weight': summary['fixed_correction_weight'],
    'gates': summary['gates'],
    'next_step': summary['next_step'],
}


In [ ]:
import pandas as pd
pd.DataFrame({
    'standard_delta': summary['standard']['fold_deltas'],
}).sort_index()


In [ ]:
pd.DataFrame(summary['diagnostic_weight_report'])


## Send back

Please send the complete summary dictionary from the validation cell. If promotion is false, also send the fold-delta table; it tells us whether to change the surface target, regional prior, or correction constraints before Stage 12.
